<a href="https://colab.research.google.com/github/Aje-dotcom/DeepTech-/blob/master/Ekene__Ajemba_Assignment_week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Setup and Data Preprocessing
First,  I'll import the necessary libraries and load the dataset. CNNs require numerical data, so I will preprocess the images by resizing them, normalizing the pixel values, and applying data augmentation to improve model generalization.

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Define dataset paths (Update these with the actual path to your dataset)
train_dir = 'path_to_your_data/train'
test_dir = 'path_to_your_data/test'

# 1. Data Augmentation and Preprocessing
# Normalizing pixel values to be between 0 and 1
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # Reserving 20% for validation
)

test_datagen = ImageDataGenerator(rescale=1./255)

# 2. Loading the data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128), # Resizing images
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Step 2: <b>Building the CNN Architecture<br></b>
Convolutional Neural Networks (CNNs) are specialized for processing grid-like data such as images. We will build the model using:  
<br><b>Convolutional Layers:</b> To extract visual features from the boat images. <I> <b>
Max Pooling Layers:</b> To reduce the spatial dimensions of the feature maps by taking the maximum value in a region. </I> <B><br>
ReLU Activation:</b> The most common default activation function used in hidden layers.  <br><b>
Softmax Activation:</b> Used in the final output layer to output probability distributions over my specific boat categories (gondola, motorboat, ferry).

In [ ]:
model = models.Sequential([
    # First Convolutional Block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D((2, 2)),

    # Second Convolutional Block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Third Convolutional Block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flattening and Fully Connected Layers
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Prevents overfitting

    # Output Layer (3 classes: gondola, motorboat, ferry)
    layers.Dense(3, activation='softmax')
])

model.summary()

Step 3:<b> Model Compilation and Training</b><br>
Since this is a multi-class classification problem, i will compile the model using <b>Categorical Cross-Entropy</b> as the loss function.

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator
)

Step 4: <b>Model Evaluation<br></b>
Finally, the assignment requires evaluating the model's accuracy, precision, and recall. I can use scikit-learn to generate a comprehensive classification report.

In [ ]:
# Generate predictions on the test set
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Get class labels
class_labels = list(test_generator.class_indices.keys())

# Print evaluation metrics
print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_labels))